In [ ]:
# 13_nl_profiles.ipynb
#
# For each row in the synthetic population, generates a natural-language
# description of that individual using the variable labels from config.
#
# Output: data/13_nl_profiles/nl_profiles.csv
#   Columns: pidp, ladcd, nl_profile
#
# Each row reads like:
#   "A 42 year old White British male whose highest qualification is a Degree.
#    They are employed, married/civil partner, owner-occupied, and live in a
#    couple household with no children."

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES

import data_pipeline.config_variables as _cv
_cv.reload_config_variables()
from data_pipeline.config_variables import VARIABLES
import data_pipeline.config_cluster as _cc
importlib.reload(_cc)
from data_pipeline.config_cluster import WAVE

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
SYNPOP_PARQUET = Path(f"../{DATA_FOLDER}/6_synthetic_population/synthetic_population.parquet")
OUTPUT_DIR     = Path(f"../{DATA_FOLDER}/13_nl_profiles")
OUTPUT_CSV     = OUTPUT_DIR / "nl_profiles.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNIT_FILTER = list(FOUR_LA_CODES) if USE_FOUR_LA_SUBSET else None

# Variables to include in the profile, in sentence order
PROFILE_VARS = ["doby_dv", "racel_dv", "sex_dv", "hiqual_dv",
                "jbstat", "marstat_dv", "tenure_dv", "hhtype_dv", "scsf1"]


# Read only the columns we need — always the raw pre-engineering column
cols_to_read = ["pidp", "ladcd"] + [f"{WAVE}_{v}" for v in PROFILE_VARS]

print(f"Reading: {SYNPOP_PARQUET}")
df = pd.read_parquet(SYNPOP_PARQUET, columns=cols_to_read)
print(f"Loaded {len(df):,} rows")

# Apply LA filter
if UNIT_FILTER:
    df = df[df["ladcd"].isin(set(UNIT_FILTER))]
    print(f"Filtered to {len(df):,} rows ({len(UNIT_FILTER)} LAs)")


In [ ]:
# ── Label decoders ────────────────────────────────────────────────────────────

_CURRENT_YEAR = pd.Timestamp.now().year

_SKIP_LABELS = {"not provided", "not applicable", "inapplicable", "missing",
                "refusal", "don't know", "proxy", "not relevant", "no answer"}


def decode(base_code: str, val) -> str | None:
    """Return the full original label for a raw UKHLS category code, or None.
    Uses VARIABLES[base]['categories'] directly — no remapping applied.
    For doby_dv (birth year), computes and returns age as a string."""
    if pd.isna(val):
        return None
    fval = float(val)
    if fval < 0:
        return None  # -1 sentinel from 3a backfill = not provided
    if base_code == "doby_dv":
        age = _CURRENT_YEAR - int(fval)
        return str(age) if 0 < age < 120 else None
    var_def = VARIABLES.get(base_code, {})
    cat_map = var_def.get("categories")
    if cat_map:
        label = cat_map.get(fval)
        if label and label.lower().strip() not in _SKIP_LABELS:
            return label
        return None
    return str(int(round(fval)))


def build_profile(row: pd.Series) -> str:
    """Build a natural-language profile string from one synthetic-population row."""
    age        = decode("doby_dv",   row.get(f"{WAVE}_doby_dv"))
    ethnicity  = decode("racel_dv",  row.get(f"{WAVE}_racel_dv"))
    sex        = decode("sex_dv",    row.get(f"{WAVE}_sex_dv"))
    hiqual     = decode("hiqual_dv", row.get(f"{WAVE}_hiqual_dv"))
    jbstat     = decode("jbstat",    row.get(f"{WAVE}_jbstat"))
    marstat    = decode("marstat_dv",row.get(f"{WAVE}_marstat_dv"))
    tenure     = decode("tenure_dv", row.get(f"{WAVE}_tenure_dv"))
    hhtype     = decode("hhtype_dv", row.get(f"{WAVE}_hhtype_dv"))
    health     = decode("scsf1",     row.get(f"{WAVE}_scsf1"))

    # Sentence 1: demographics
    age_str = f"{age} year old" if age else "unknown age"
    eth_str = f"{ethnicity} " if ethnicity else ""
    sex_str = sex.lower() if sex else "person"
    s1 = f"A {age_str} {eth_str}{sex_str}"

    # Sentence 2: qualification
    s2 = f"whose highest qualification is {hiqual.lower()}" if hiqual else None

    # Sentence 3: employment
    s3 = f"Their employment status is {jbstat.lower()}" if jbstat else None

    # Sentence 4: marital status
    s4 = f"their marital status is {marstat.lower()}" if marstat else None

    # Sentence 5: housing
    s5 = f"their housing tenure is {tenure.lower()}" if tenure else None

    # Sentence 6: household type
    s6 = f"and they live in a {hhtype.lower()} household" if hhtype else None

    # Sentence 7: health
    s7 = f"They rate their general health as {health.lower()}" if health else None

    # Assemble
    middle = ", ".join(p for p in [s2, s3, s4, s5, s6] if p)
    profile = s1
    if middle:
        profile += f", {middle}."
    if s7:
        profile += f" {s7}."

    return profile


# Sanity check on a sample row
sample = df.iloc[0]
print("Sample profile:")
print(build_profile(sample))


In [ ]:
# ── Generate profiles ─────────────────────────────────────────────────────────

tqdm.pandas(desc="Building profiles")
df["nl_profile"] = df.progress_apply(build_profile, axis=1)

out = df[["pidp", "ladcd", "nl_profile"]].copy()

out.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {len(out):,} rows to {OUTPUT_CSV}")
display(out.head(5))
